# 01.9 阶段项目：表格分类

这一份 notebook 是 `Phase 1` 的综合小项目。  

目标不是追求最强模型，而是把前面学的东西完整串起来：  

- 数据读取（data loading）
- 训练集与测试集划分（train-test split）
- 特征标准化（feature standardization）
- 张量转换（tensor conversion）
- `Dataset` 与 `DataLoader`
- MLP 模型（MLP model）
- 训练与评估（training and evaluation）
- 最终推理（final inference）

## 学习目标

学完后你应该能

1. 完成一个最小端到端分类项目
2. 用 `PyTorch` 处理表格数据
3. 理解训练、验证、测试这三个阶段
4. 用 loss 和 accuracy 监控训练
5. 对新样本做预测
6. 为后面更大项目打基础

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

## 1. 读取数据

这里使用 `iris` 数据集，因为它足够小，适合作为第一个完整项目。  


In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()

print(df.head())
print()
print("shape =", df.shape)
print("target names =", iris.target_names)

## 2. 划分训练、验证、测试

这里采用两次切分：  

1. 先切出测试集
2. 再从训练集中切出验证集

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=y_train_full,
)

print("train shape =", X_train.shape)
print("val shape =", X_val.shape)
print("test shape =", X_test.shape)

## 3. 标准化特征

标准化器只在训练集上 `fit`，然后应用到验证集和测试集。  


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled[:2])

## 4. 转成张量并构建 DataLoader

pandas` 世界接到 `PyTorch` 世界。  


In [ ]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.long)
y_val_tensor = torch.tensor(y_val.to_numpy(), dtype=torch.long)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
val_ds = TensorDataset(X_val_tensor, y_val_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

xb, yb = next(iter(train_loader))
print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)

## 5. 定义模型

因为 `iris` 有 4 个特征和 3 个类别，所以这里用：  

- 输入维度（input dimension: 4）
- 输出维度（output dimension: 3）

In [ ]:
class IrisMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 3),
        )

    def forward(self, x):
        return self.net(x)


model = IrisMLP()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)

print(model)

## 6. 定义训练与评估函数

这里把训练逻辑封装成函数，后面项目会更清晰。  


In [ ]:
def batch_accuracy(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_acc += batch_accuracy(logits, yb)
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches

## 7. 开始训练

这里训练 40 个 epoch。  


In [ ]:
history = []

for epoch in range(1, 41):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
    )

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"epoch={epoch:02d} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

In [ ]:
history_df = pd.DataFrame(history)
print(history_df.tail())

## 8. 在测试集上评估

训练和调参看验证集，最后只在测试集上做一次最终评估。  


In [ ]:
model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb)
        preds = logits.argmax(dim=1)
        all_preds.append(preds)
        all_targets.append(yb)

all_preds = torch.cat(all_preds)
all_targets = torch.cat(all_targets)

test_acc = accuracy_score(all_targets.numpy(), all_preds.numpy())
cm = confusion_matrix(all_targets.numpy(), all_preds.numpy())

print("test accuracy =", test_acc)
print("confusion matrix =\n", cm)

## 9. 对新样本做推理

真正落地时，模型的价值之一就是能对新数据做预测。  


In [ ]:
new_samples = pd.DataFrame(
    {
        "sepal length (cm)": [5.0, 6.5],
        "sepal width (cm)": [3.4, 3.0],
        "petal length (cm)": [1.5, 5.5],
        "petal width (cm)": [0.2, 2.0],
    }
)

new_scaled = scaler.transform(new_samples)
new_tensor = torch.tensor(new_scaled, dtype=torch.float32)

with torch.no_grad():
    logits = model(new_tensor)
    preds = logits.argmax(dim=1)

pred_names = [iris.target_names[i] for i in preds.tolist()]
print(preds)
print(pred_names)

## 10. 小练习

这几题主要是为了让你开始主动做小实验。  


In [ ]:
# 练习 1
# 把隐藏层宽度从 16 改成 32，再训练一遍，比较验证集准确率。
# Change the hidden width from 16 to 32, train again, and compare validation accuracy.

In [ ]:
# 练习 2
# 把优化器从 Adam 改成 SGD，再观察训练速度和最终结果。
# Change the optimizer from Adam to SGD and observe training speed and final results.

In [ ]:
# 练习 3
# 用一句话说明为什么标准化器只能在训练集上 fit。
# In one sentence, explain why the scaler should only be fit on the training set.

参考回答

因为如果标准化器先看到了验证集或测试集的信息，就会造成数据泄漏  


## 11. 小结

到这里，你已经完成了一个最小但完整的 `PyTorch` 表格分类项目。  

你已经串起来的内容

- 数据读取（data loading）
- 划分数据集（dataset splitting）
- 特征预处理（feature preprocessing）
- `TensorDataset` 和 `DataLoader`
- 模型定义（model definition）
- 训练循环（training loop）
- 测试评估（test evaluation）
- 推理（inference）

这就是后面更大项目的缩小版。  

下一步建议

- `Phase 1` 已经可以视为一个完整单元，接下来可以进入 CNN